# **Квазиэксперименты: разность разностей, синтетический контроль и разрывный дизайн**

Практика к модулю [«Каузальность и линейные модели»](https://ai-interpretability.school/learn/3/0/0), урок «Квазиэксперименты».

В уроке схемы были формулами и задачами на таблицах из нескольких чисел. Здесь мы посчитаем их на настоящих данных и на синтетической панели, где истинный эффект известен. Проверим пять утверждений урока:

1. разность разностей держится на параллельных трендах, и их нарушение до начала воздействия видно на данных;
2. синтетический контроль подбирает веса контрольных объектов так, чтобы они повторили траекторию объекта до воздействия;
3. synthetic DiD соединяет веса объектов, веса периодов и разность разностей;
4. регрессия с фиксированными эффектами (TWFE) смещена, когда группы получают воздействие в разное время и эффект растет со временем, а оценка Callaway и Sant'Anna — нет;
5. разрывный дизайн оценивает скачок исхода на пороге, и ответ зависит от ширины окна.

План тетради:

1. данные Proposition 99;
2. разность разностей;
3. синтетический контроль;
4. synthetic DiD;
5. разность разностей с разным моментом начала воздействия;
6. разрывный дизайн на выборах в Сенат США;
7. где ломается;
8. те же оценки в библиотеках.

Тетрадь идет на процессоре, прогон занимает около минуты.

In [ ]:
%pip install -q rdrobust==2.1.1 csdid==0.4.2 drdid==1.1.6 pysyncon==1.7.0

In [ ]:
import warnings
import numpy as np
import pandas as pd
import cvxpy as cp
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf

warnings.filterwarnings('ignore')
SEED = 0
pd.set_option('display.width', 140)

## 1. Данные: Proposition 99

В 1988 году Калифорния приняла Proposition 99: акциз на сигареты вырос на 25 центов за пачку, и деньги пошли на программы против курения. Закон действует с 1989 года. Классический разбор — [Abadie, Diamond и Hainmueller, 2010](https://doi.org/10.1198/jasa.2009.ap08746); мы берем таблицу из пакета [synthdid](https://github.com/synth-inference/synthdid) (лицензия BSD-3), на которой считали оценки [Arkhangelsky et al., 2021](https://doi.org/10.1257/aer.20190159).

В таблице 39 штатов за 1970–2000 годы: Калифорния и 38 штатов, где таких программ и крупных повышений акциза в эти годы не было. Исход `PacksPerCapita` — продажи сигарет, пачек на душу населения в год. Периодов до воздействия 19 (1970–1988), после — 12 (1989–2000).

In [ ]:
url = 'https://github.com/SadSabrina/open-xai-materials/raw/refs/heads/main/data/california_prop99.csv'
prop99 = pd.read_csv(url, sep=';')
Y = prop99.pivot(index='State', columns='Year', values='PacksPerCapita')
treated = 'California'
controls = [s for s in Y.index if s != treated]
years = Y.columns.values
pre, post = years < 1989, years >= 1989
y_tr = Y.loc[treated].values          # траектория Калифорнии
Y_co = Y.loc[controls].values         # 38 контрольных штатов x 31 год
print(Y.shape, len(controls), pre.sum(), post.sum())

plt.figure(figsize=(8, 3.5))
plt.plot(years, y_tr, label='California')
plt.plot(years, Y_co.mean(0), label='mean of 38 states')
plt.axvline(1988.5, color='gray', ls='--')
plt.ylabel('packs per capita'); plt.legend(); plt.show()

Продажи падают везде, а в Калифорнии быстрее, и разрыв растет еще до 1989 года. Это первое, что надо держать в голове, читая оценки ниже.

## 2. Разность разностей

$$\hat\tau_{DiD} = (\bar Y_{T,\text{after}} - \bar Y_{T,\text{before}}) - (\bar Y_{C,\text{after}} - \bar Y_{C,\text{before}}),$$

где $T$ — Калифорния, $C$ — 38 контрольных штатов, before — 1970–1988, after — 1989–2000.

**Задание 1. Посчитайте оценку разности разностей для Калифорнии: по формуле из средних и коэффициентом при `treated:post` в регрессии `PacksPerCapita ~ treated * post` по всем 39 штатам и 31 году. Сравните ее с двумя наивными оценками: разностью Калифорнии и среднего по 38 штатам после 1989 года и разностью Калифорнии после и до 1989 года. Ответ округлите до двух знаков после точки.**

In [ ]:
# Ваш код здесь

Обе наивные оценки больше по модулю: первая смешивает эффект с тем, что Калифорния и раньше курила меньше, вторая — с общим падением продаж во всех штатах. Разность разностей убирает обе помехи, если без закона разрыв между Калифорнией и средним по штатам оставался бы постоянным. Насколько это так, проверим в разделе 7.

## 3. Синтетический контроль

Вместо среднего по всем 38 штатам синтетический контроль строит взвешенную смесь штатов, которая повторяет траекторию Калифорнии до 1989 года:

$$\hat w = \arg\min_{w \ge 0,\ \sum_j w_j = 1}\ \sum_{t \le 1988}\Big(Y_{\text{CA},t} - \sum_j w_j Y_{j,t}\Big)^2,\qquad \hat\tau_t = Y_{\text{CA},t} - \sum_j \hat w_j Y_{j,t}.$$

У [Abadie, Diamond и Hainmueller, 2010](https://doi.org/10.1198/jasa.2009.ap08746) в подгонке участвуют еще предикторы (доход, цена сигарет, доля молодежи) с весами $V$; мы, как [Arkhangelsky et al., 2021](https://doi.org/10.1257/aer.20190159), подгоняем только по прошлым значениям исхода. Задача квадратичная с ограничениями, ее решает `cvxpy`. Малый L2-штраф ($\zeta = 10^{-6}\hat\sigma$, как в той же статье) делает решение единственным.

In [ ]:
def sc_weights(y_target, Y_donors, zeta=1e-6):
    """Веса синтетического контроля: неотрицательные, в сумме 1, подгонка по периодам y_target."""
    w = cp.Variable(Y_donors.shape[0])
    objective = cp.sum_squares(Y_donors.T @ w - y_target) + zeta ** 2 * len(y_target) * cp.sum_squares(w)
    cp.Problem(cp.Minimize(objective), [w >= 0, cp.sum(w) == 1]).solve(solver=cp.CLARABEL)
    return np.clip(w.value, 0, None)


w_sc = sc_weights(y_tr[pre], Y_co[:, pre])
weights = pd.Series(w_sc, index=controls).sort_values(ascending=False)
print(weights[weights > 0.01].round(3))
synthetic = w_sc @ Y_co
rmspe_sc = np.sqrt(((y_tr - synthetic)[pre] ** 2).mean())
level_did = Y_co.mean(0) + (y_tr[pre] - Y_co[:, pre].mean(0)).mean()   # среднее по штатам, сдвинутое на разрыв до 1989
rmspe_did = np.sqrt(((y_tr - level_did)[pre] ** 2).mean())
print(f'RMSPE до 1989: синтетический контроль {rmspe_sc:.2f} | среднее 38 штатов со сдвигом {rmspe_did:.2f}')

plt.figure(figsize=(8, 3.5))
plt.plot(years, y_tr, label='California')
plt.plot(years, synthetic, label='synthetic California')
plt.axvline(1988.5, color='gray', ls='--')
plt.ylabel('packs per capita'); plt.legend(); plt.show()

Ненулевой вес получили шесть штатов, больше всего — Юта, Монтана и Невада. До 1989 года синтетическая Калифорния отклоняется от настоящей в среднем на 1.66 пачки, а среднее по 38 штатам, сдвинутое на разницу уровней, — на 7.16: смесь повторяет не только уровень, но и наклон.

**Задание 2. Посчитайте оценку эффекта синтетическим контролем — средний разрыв между Калифорнией и синтетической Калифорнией за 1989–2000 годы. Ответ округлите до одного знака после точки.**

In [ ]:
# Ваш код здесь

В статье Arkhangelsky et al. та же оценка равна −19.6; разница в 0.1 — от решателя: веса синтетического контроля без регуляризации определены неединственно, о чем авторы пишут в сноске 5.

**Плацебо по штатам.** Стандартных ошибок у синтетического контроля с одним объектом воздействия нет. Abadie et al. предлагают по очереди «назначить» закон каждому контрольному штату, построить для него синтетический контроль и сравнить отношение ошибки после 1989 года к ошибке до него. Если у Калифорнии отношение самое большое из 39, вероятность случайно получить такой результат — $1/39 \approx 0.026$.

In [ ]:
ratio = {}
for s in Y.index:
    y_s, donors = Y.loc[s].values, Y.drop(index=s).values
    gap = y_s - sc_weights(y_s[pre], donors[:, pre]) @ donors
    ratio[s] = np.sqrt((gap[post] ** 2).mean()) / np.sqrt((gap[pre] ** 2).mean())
ratio = pd.Series(ratio).sort_values(ascending=False)
rank = list(ratio.index).index(treated) + 1
print(ratio.head(5).round(2))
print(f'Калифорния: место {rank} из {len(ratio)}, плацебо-p = {rank / len(ratio):.3f}')

В нашей спецификации — только прошлые значения исхода, без предикторов — Калифорния третья из 39, и плацебо-p равно 0.077. У Abadie et al. с предикторами она первая. Вывод о значимости зависит от спецификации, а сама оценка эффекта — нет: она в обоих случаях около −19 пачек.

## 4. Synthetic DiD

[Arkhangelsky et al., 2021](https://doi.org/10.1257/aer.20190159) соединяют две идеи. Веса штатов $\hat\omega$ подбирают как в синтетическом контроле, но с константой $\omega_0$ — совпасть с Калифорнией нужно по форме траектории, а не по уровню — и с L2-штрафом $\zeta^2 T_{pre}\|\omega\|^2$, где $\zeta = (N_{tr}T_{post})^{1/4}\hat\sigma$, а $\hat\sigma$ — стандартное отклонение годовых приращений исхода в контрольных штатах до 1989 года. Веса лет $\hat\lambda$ подбирают так, чтобы взвешенное среднее лет до воздействия у контрольных штатов повторяло их среднее после, тоже с константой. Оценка — разность разностей с этими весами:

$$\hat\tau_{SDID} = \Big(\bar Y_{\text{CA},post} - \sum_{t\le 1988}\hat\lambda_t Y_{\text{CA},t}\Big) - \sum_j \hat\omega_j\Big(\bar Y_{j,post} - \sum_{t\le 1988}\hat\lambda_t Y_{j,t}\Big).$$

Веса считает код ниже, сама оценка — задание 3.

In [ ]:
T_pre, T_post, N_co = pre.sum(), post.sum(), len(controls)
sigma = np.diff(Y_co[:, pre], axis=1).std(ddof=1)
zeta = (1 * T_post) ** 0.25 * sigma

omega0, omega = cp.Variable(), cp.Variable(N_co)
cp.Problem(cp.Minimize(cp.sum_squares(omega0 + Y_co[:, pre].T @ omega - y_tr[pre]) + zeta ** 2 * T_pre * cp.sum_squares(omega)),
           [omega >= 0, cp.sum(omega) == 1]).solve(solver=cp.CLARABEL)
omega = np.clip(omega.value, 0, None)

lam0, lam = cp.Variable(), cp.Variable(T_pre)
cp.Problem(cp.Minimize(cp.sum_squares(lam0 + Y_co[:, pre] @ lam - Y_co[:, post].mean(1)) + (1e-6 * sigma) ** 2 * N_co * cp.sum_squares(lam)),
           [lam >= 0, cp.sum(lam) == 1]).solve(solver=cp.CLARABEL)
lam = np.clip(lam.value, 0, None)

print(f'sigma {sigma:.3f} | zeta {zeta:.3f}')
print(f'штатов с весом больше 0.01: {(omega > 0.01).sum()} из {N_co}, наибольший вес {omega.max():.3f}')
print('веса лет:', {int(y): float(round(v, 3)) for y, v in zip(years[pre], lam) if v > 0.001})

Штрафом веса штатов распределены: 22 штата с весом больше 0.01 и ни одного больше 0.13. Веса лет легли на 1986–1988 годы: для сравнения «до» берутся годы, похожие на период после воздействия.

**О решателе.** Тетрадь решает обе задачи о весах точно, через `cvxpy`. R-пакет `synthdid` авторов статьи по умолчанию решает их иначе: методом Франк–Вулфа ([`sc.weight.fw` в `R/solver.R`](https://github.com/synth-inference/synthdid/blob/master/R/solver.R)) с остановкой по `min.decrease` и `max.iter`, а после первых 100 итераций разреживает веса и продолжает с них ([`sparsify_function` в `R/synthdid.R`](https://github.com/synth-inference/synthdid/blob/master/R/synthdid.R)): веса не больше четверти наибольшего зануляются, остальные нормируются к сумме 1. Поэтому число «22 штата с весом больше 0.01» — свойство решения этой тетради, а не `synthdid`.

**Задание 3. Посчитайте оценку synthetic DiD по формуле выше из весов `omega` и `lam`. Ответ округлите до одного знака после точки.**

In [ ]:
# Ваш код здесь

Три оценки на одних данных рядом с таблицей 1 статьи Arkhangelsky et al. (в статье: DiD −27.3, синтетический контроль −19.6, SDID −15.6):

In [ ]:
did_m = (y_tr[post].mean() - y_tr[pre].mean()) - (Y_co[:, post].mean() - Y_co[:, pre].mean())
sc_m = (y_tr - w_sc @ Y_co)[post].mean()
sdid_m = (y_tr[post].mean() - lam @ y_tr[pre]) - omega @ (Y_co[:, post].mean(1) - Y_co[:, pre] @ lam)
pd.DataFrame({'наша оценка': [did_m, sc_m, sdid_m], 'в статье': [-27.3, -19.6, -15.6]},
             index=['DiD', 'синтетический контроль', 'synthetic DiD']).round(2)

## 5. Разность разностей с разным моментом начала воздействия

Теперь синтетическая панель, где истинный эффект известен. 300 объектов наблюдаются 10 периодов: 100 получают воздействие с периода 4, 100 — с периода 7, 100 не получают никогда. Эффект растет со временем: в первый период воздействия он равен 1, дальше прибавляет 0.5 за период,

$$Y_{it} = a_i + 0.3\,t + \tau_{it} + \varepsilon_{it},\qquad \tau_{it} = \big(1 + 0.5\,(t - g_i)\big)\,\mathbf 1[t \ge g_i],$$

где $a_i$ — постоянное различие объектов (у ранней группы в среднем на 0.5 выше, чем у группы без воздействия, у поздней — на 0.2), $g_i$ — период начала воздействия, $\varepsilon_{it} \sim \mathcal N(0, 0.5^2)$. Тренды без воздействия параллельны по построению.

In [ ]:
def make_panel(slope, seed=SEED):
    rng = np.random.default_rng(seed)
    rows, uid = [], 0
    for g, n, shift in ((4, 100, 0.5), (7, 100, 0.2), (0, 100, 0.0)):   # g = 0: никогда не получают
        for _ in range(n):
            a = rng.normal() + shift
            for t in range(1, 11):
                d = int(g > 0 and t >= g)
                tau = (1 + slope * (t - g)) if d else 0.0
                rows.append((uid, t, g, d, a + 0.3 * t + tau + rng.normal(0, 0.5), tau))
            uid += 1
    return pd.DataFrame(rows, columns=['id', 't', 'g', 'd', 'y', 'tau'])


panel = make_panel(slope=0.5)
true_att = panel.loc[panel.d == 1, 'tau'].mean()
print(f'наблюдений {len(panel)} | истинный эффект, среднее по наблюдениям с воздействием: {true_att:.3f}')

**Задание 4. Оцените эффект регрессией с фиксированными эффектами объекта и периода (TWFE): `y ~ d + C(id) + C(t)`, где `d` — индикатор «объект уже получает воздействие». Сравните коэффициент при `d` с истинным эффектом. Ответ округлите до трех знаков после точки.**

In [ ]:
# Ваш код здесь

Откуда смещение, показывает разложение [Goodman-Bacon, 2021](https://doi.org/10.1016/j.jeconom.2021.03.014): коэффициент TWFE — взвешенное среднее всех сравнений «две группы, два периода», и среди них есть сравнение поздней группы с ранней, которая к этому времени уже получает воздействие. Посчитаем его отдельно: периоды 4–6 против 7–10, поздняя группа против ранней.

In [ ]:
m = panel.groupby(['g', 't']).y.mean().unstack()
late_vs_early = (m.loc[7, 7:10].mean() - m.loc[7, 4:6].mean()) - (m.loc[4, 7:10].mean() - m.loc[4, 4:6].mean())
true_late = panel.loc[(panel.g == 7) & (panel.d == 1), 'tau'].mean()
print(f'поздняя против ранней, уже получившей воздействие: {late_vs_early:.3f} | истинный эффект поздней группы в периоды 7-10: {true_late:.3f}')

Эффект ранней группы за эти периоды вырос, и сравнение вычитает этот рост из эффекта поздней группы: от истинного эффекта 1.75 остается почти ноль.

[Callaway и Sant'Anna, 2021](https://doi.org/10.1016/j.jeconom.2020.12.001) запрещают такие сравнения. Для каждой группы $g$ и периода $t \ge g$ они считают отдельный эффект как разность разностей с периодом $g-1$ и с группой, которая не получает воздействия никогда:

$$ATT(g, t) = E[Y_t - Y_{g-1} \mid G = g] - E[Y_t - Y_{g-1} \mid G = \infty],$$

а потом усредняют эффекты $ATT(g,t)$ с весами, пропорциональными размерам групп.

**Задание 5. Посчитайте $ATT(g,t)$ для групп 4 и 7 и всех периодов $t \ge g$ по средним `m` и усредните их с весами по размеру группы. Сравните с истинным эффектом и с TWFE. Ответ округлите до трех знаков после точки.**

In [ ]:
# Ваш код здесь

## 6. Разрывный дизайн: выборы в Сенат США

Данные [Cattaneo, Frandsen и Titiunik, 2015](https://doi.org/10.1515/jci-2013-0010) идут вместе с пакетом `rdrobust` (лицензия GPL-3.0): 1390 выборов в Сенат США, у 1297 из них известны и отрыв, и исход на следующих выборах. Running variable `margin` — отрыв кандидата-демократа на выборах $t$ в процентных пунктах, порог — 0: при отрыве выше нуля демократ побеждает. Исход `vote` — доля голосов демократа на следующих выборах на то же место в Сенате через шесть лет. Скачок на пороге — преимущество партии, которая уже держит место (incumbency advantage), как в [Lee, 2008](https://doi.org/10.1016/j.jeconom.2007.05.004).

In [ ]:
from rdrobust import rdrobust
from rdrobust.datasets import rdrobust_RDsenate

senate = rdrobust_RDsenate().dropna(subset=['vote', 'margin'])
x, y = senate.margin.values, senate.vote.values
print(f'выборов с обоими значениями: {len(senate)}')
print(f'наивная разность средних справа и слева от порога: {y[x >= 0].mean() - y[x < 0].mean():.2f}')

rd = rdrobust(y=y, x=x)
h = rd.bws.loc['h', 'left']
print(f'rdrobust: оценка {rd.coef.loc["Conventional", "Coeff"]:.3f} | окно h = {h:.3f}')

Наивная разность — 21.70 пункта: справа от порога собраны штаты, где демократы сильны, слева — где слабы. `rdrobust` по умолчанию строит по обе стороны порога линейную регрессию в окне $h$, которое подбирает по оценке среднеквадратичной ошибки ([Calonico, Cattaneo и Titiunik, 2014](https://doi.org/10.3982/ECTA11757)), с треугольным ядром: вес объекта $\max(0,\ 1 - |x|/h)$.

**Задание 6. Повторите оценку `rdrobust` руками: в окне $|x| \le h$ постройте по каждую сторону порога взвешенную регрессию `vote` на `margin` с весами $1 - |x|/h$ (`sm.WLS`) и возьмите разность свободных членов. Ответ округлите до двух знаков после точки.**

In [ ]:
# Ваш код здесь

## 7. Где ломается

### 7.1. Параллельные тренды до воздействия

Проверка без данных после воздействия: притворимся, что закон приняли в 1980 году, и посчитаем разность разностей только по 1970–1988 годам. Эффекта в эти годы не было, и при параллельных трендах оценка была бы около нуля.

In [ ]:
fake = years >= 1980
did_placebo = (y_tr[pre & fake].mean() - y_tr[pre & ~fake].mean()) - (Y_co[:, pre & fake].mean() - Y_co[:, pre & ~fake].mean())
print(f'плацебо-DiD с фиктивным началом в 1980 году: {did_placebo:.2f}')

Калифорния снижала продажи быстрее остальных и до закона, и разность разностей приписывает часть этого тренда закону. Синтетический контроль и SDID подбирают сравнение по траектории до воздействия, поэтому их оценки меньше по модулю.

### 7.2. TWFE смещена только при меняющемся эффекте

Та же панель, но эффект постоянный (наклон 0) и растущий быстрее (наклон 1):

In [ ]:
rows = []
for slope in (0.0, 0.5, 1.0):
    p = make_panel(slope)
    rows.append(dict(slope=slope, true=p.loc[p.d == 1, 'tau'].mean(),
                     TWFE=smf.ols('y ~ d + C(id) + C(t)', data=p).fit().params['d']))
pd.DataFrame(rows).round(3)

При постоянном эффекте TWFE почти не смещена: сравнение с уже получившей воздействие группой вычитает только ее постоянный эффект, а он сокращается. Чем быстрее растет эффект, тем больше смещение.

### 7.3. Окно разрывного дизайна

Оценка при разных ширинах окна, обычная линейная регрессия по каждую сторону без ядра:

In [ ]:
def rd_uniform(width):
    inside = np.abs(x) <= width
    fit = lambda mask: sm.OLS(y[mask], sm.add_constant(x[mask])).fit().params[0]
    return fit(inside & (x >= 0)) - fit(inside & (x < 0)), inside.sum()

pd.DataFrame([(w_, *rd_uniform(w_)) for w_ in (5, 10, 20, 40, 100)], columns=['h', 'jump', 'n']).round(3)

Узкое окно (5 пунктов, 245 выборов) дает 9.8, широкое (все 1297 выборов) — 6.0. Узкое окно ближе к сравнению «почти одинаковых» выборов, но шумнее; широкое опирается на линейность далеко от порога. Поэтому оценку разрывного дизайна показывают при нескольких окнах.

Вторая проверка — плотность running variable у порога: если бы отрыв можно было подогнать, выборов чуть выше нуля было бы больше, чем чуть ниже. Тест McCrary делает это строго, здесь — грубый подсчет.

In [ ]:
for b in (1, 2, 5):
    print(f'отрыв в [-{b}, 0): {((x >= -b) & (x < 0)).sum():4d} | в [0, {b}): {((x >= 0) & (x < b)).sum():4d}')

Скачка числа выборов на пороге нет: 49 выборов с отрывом от −2 до 0 против 47 с отрывом от 0 до 2.

## 8. Те же оценки в библиотеках

Синтетический контроль Abadie et al. — пакет [`pysyncon`](https://github.com/sdfordham/pysyncon) (лицензия MIT). Он подбирает и веса штатов $W$, и веса предикторов $V$ вложенной оптимизацией; в предикторы передаем те же 19 значений исхода до 1989 года. Оценка Callaway и Sant'Anna — пакет [`csdid`](https://github.com/d2cml-ai/csdid) (лицензия MIT).

In [ ]:
from pysyncon import Dataprep, Synth

years_pre = list(range(1970, 1989))
dataprep = Dataprep(foo=prop99, predictors=[], predictors_op='mean', time_predictors_prior=years_pre,
                    special_predictors=[('PacksPerCapita', [yr], 'mean') for yr in years_pre],
                    dependent='PacksPerCapita', unit_variable='State', time_variable='Year',
                    treatment_identifier=treated, controls_identifier=controls, time_optimize_ssr=years_pre)
synth = Synth()
synth.fit(dataprep=dataprep)
Z0, Z1 = dataprep.make_outcome_mats(time_period=range(1989, 2001))
print(synth.weights()[synth.weights() > 0.01].round(2))
print(f'pysyncon: {(Z1 - Z0 @ synth.W).mean():.1f} | наш синтетический контроль: {(y_tr - w_sc @ Y_co)[post].mean():.1f}')

Те же штаты с близкими весами и оценка около −19. Число зависит от вложенной оптимизации весов $V$ и может немного отличаться от прогона к прогону и между машинами.

In [ ]:
from csdid.att_gt import ATTgt

cs = ATTgt(yname='y', gname='g', idname='id', tname='t', data=panel).fit(est_method='reg', bstrap=False)
cs.aggte(typec='simple', bstrap=False)
print(f"csdid: {cs.atte['overall_att']:.3f}")

`csdid` повторяет усреднение $ATT(g,t)$ из задания 5 и печатает таблицу $ATT(g,t)$ со стандартными ошибками. Разрывный дизайн мы уже считали библиотекой `rdrobust` в разделе 6.

## Что унести из тетради

- **Разность разностей проверяют до воздействия.** Плацебо с фиктивным началом в 1980 году дает −11.51 пачки: тренды не параллельны, и оценка −27.35 завышена по модулю.
- **Синтетический контроль подбирает сравнение по траектории.** Шесть штатов с весом повторяют Калифорнию до 1989 года с ошибкой 1.66 пачки против 7.16 у среднего по штатам; оценка −19.5. Вывод о значимости по плацебо зависит от спецификации.
- **Synthetic DiD соединяет обе идеи.** Веса штатов с константой и штрафом, веса лет, разность разностей с весами — оценка −15.6, как в статье.
- **TWFE при разном моменте начала воздействия смещена, если эффект меняется во времени.** 1.662 при истинном эффекте 2.227; виновато сравнение с группой, которая уже получила воздействие. Оценка Callaway и Sant'Anna (2.297) таких сравнений не делает.
- **Разрывный дизайн оценивает эффект у порога.** 7.41 пункта при наивной разности средних 21.70; ответ зависит от окна, и его показывают при нескольких окнах.

Источники:
1. [Abadie, Diamond и Hainmueller, 2010. Synthetic Control Methods for Comparative Case Studies: Estimating the Effect of California's Tobacco Control Program](https://doi.org/10.1198/jasa.2009.ap08746)
2. [Arkhangelsky, Athey, Hirshberg, Imbens и Wager, 2021. Synthetic Difference-in-Differences](https://doi.org/10.1257/aer.20190159)
3. [Goodman-Bacon, 2021. Difference-in-differences with variation in treatment timing](https://doi.org/10.1016/j.jeconom.2021.03.014)
4. [Callaway и Sant'Anna, 2021. Difference-in-Differences with multiple time periods](https://doi.org/10.1016/j.jeconom.2020.12.001)
5. [Lee, 2008. Randomized experiments from non-random selection in U.S. House elections](https://doi.org/10.1016/j.jeconom.2007.05.004)
6. [Cattaneo, Frandsen и Titiunik, 2015. Randomization Inference in the Regression Discontinuity Design: An Application to Party Advantages in the U.S. Senate](https://doi.org/10.1515/jci-2013-0010)
7. [Calonico, Cattaneo и Titiunik, 2014. Robust Nonparametric Confidence Intervals for Regression-Discontinuity Designs](https://doi.org/10.3982/ECTA11757)